# Baseline

Наша задача - посмотреть, что именно вносит мультимодальность и какие методы наиболее пригодны для улучшения метрик.

Для этого нам необходимо понимать бейзлайны по разным модальностям.

In [2]:
%%capture
!pip install catboost transformers torch torchvision timm

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Шаг 1: Загрузка данных и первичный осмотр

In [4]:
#from google.colab import drive
#drive.mount('/content/drive')

In [5]:
#df = pd.read_csv('/content/drive//MyDrive/ozon_contest/ozon_train.csv', encoding='utf-8')
import pandas as pd
df = pd.read_csv('/Users/sofya/Desktop/diplomahse/ozon_train.csv')
df.head()

,id,resolution,brand_name,description,name_rus,CommercialTypeName4,rating_1_count,rating_2_count,rating_3_count,rating_4_count,...,ExemplarReturnedCountTotal30,ExemplarReturnedCountTotal90,ExemplarReturnedValueTotal7,ExemplarReturnedValueTotal30,ExemplarReturnedValueTotal90,ItemVarietyCount,ItemAvailableCount,seller_time_alive,ItemID,SellerID
0,159385,0,ACTRUM,"Мешки пылесборники для пылесоса PHILIPS, 10 шт...","Мешки для пылесоса PHILIPS TRIATLON, синтетиче...",Пылесборник,6.0,4.0,4.0,3.0,...,11.0,50.0,730.171845,896.528847,1043.118191,1.0,1.0,1860.0,78312,1218
1,288616,0,Red Line,Защитная силиконовая крышка обьектива GoPro He...,Защитная крышка Redline на экшн-камеру GoPro (...,Крышка для объектива,NaN,NaN,NaN,NaN,...,26.0,54.0,993.043882,1137.421611,1188.608000,1.0,1.0,1757.0,141999,1374
2,108090,0,Talwar Brothers,Плоский медиатор из кости толщиной 0.6 мм<br/>...,Медиатор для гитары Acura GP-PB6,Аксессуар для музыкального инструмента,0.0,0.0,1.0,0.0,...,16.0,34.0,800.822138,1174.069505,1224.798286,1.0,1.0,1722.0,53306,1448
3,415607,0,NaN,"Игра Sonic Frontiers для PlayStation 5, русски...","Игра Sonic Frontiers для PlayStation 5, русски...",Видеоигра,NaN,NaN,NaN,NaN,...,3.0,6.0,0.000000,913.530121,982.789171,3.0,3.0,1692.0,202599,715
4,332391,0,NaN,Disney Classic Games: Aladdin and The Lion Kin...,"Игра Aladdin and Lion King (PlayStation 4, анг...",Видеоигра,1.0,0.0,0.0,0.0,...,3.0,6.0,0.000000,913.542170,982.783783,3.0,3.0,1692.0,163725,715


**Наши данные**: 197,198 наблюдений, 52 признака

**Типы данных**:

- 30 числовых признаков (float64)

- 17 целочисленных признаков (int64)

- 4 текстовых признака (object)

- 1 категориальный (category)

**Ключевые группы признаков:**

- Идентификаторы: id, ItemID, SellerID

- Целевая переменная: resolution

- Временные признаки: item_time_alive, seller_time_alive

- Рейтинги и отзывы: rating_X_count, comments_published_count

- Продажи и возвраты: item_count_salesX, item_count_returnsX

- Финансовые метрики: GmvTotalX, ExemplarReturnedValueTotalX

- Текстовые данные: brand_name, description, name_rus

In [6]:
# Определяем sellers с обоими классами
seller_class_counts = df.groupby("SellerID")["resolution"].nunique()

multi_class_sellers = seller_class_counts[seller_class_counts > 1].index
single_class_sellers = seller_class_counts[seller_class_counts == 1].index

# Делим multi-class sellers
np.random.seed(42)

test_sellers = np.random.choice(
    multi_class_sellers,
    size=int(0.3 * len(multi_class_sellers)),
    replace=False
)

train_sellers = np.setdiff1d(df["SellerID"].unique(), test_sellers)

# Формируем датасеты
train_df = df[df["SellerID"].isin(train_sellers)]
test_df = df[df["SellerID"].isin(test_sellers)]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train counterfeit rate:", train_df["resolution"].mean())
print("Test counterfeit rate:", test_df["resolution"].mean())

Train shape: (177380, 45)
Test shape: (19818, 45)
Train counterfeit rate: 0.05882850377720149
Test counterfeit rate: 0.13205167019880917


In [7]:
train_df = train_df.copy()
test_df = test_df.copy()

В качестве метрик возьмем ROC-AUC и PR-AUC.

# baseline A - только текст

сначала посмотрим, какие результаты нам даст анализ исключительно текста. на озон капе это принесло неплохие результаты

In [8]:
train_df["text"] = (
    train_df["name_rus"].fillna("") + " " +
    train_df["description"].fillna("") + " " +
    train_df["brand_name"].fillna("")
)

test_df["text"] = (
    test_df["name_rus"].fillna("") + " " +
    test_df["description"].fillna("") + " " +
    test_df["brand_name"].fillna("")
)

In [9]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=5
)

X_train_text = vectorizer.fit_transform(train_df["text"])
X_test_text = vectorizer.transform(test_df["text"])

y_train = train_df["resolution"]
y_test = test_df["resolution"]

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_text, y_train)

y_pred_proba = model.predict_proba(X_test_text)[:,1]

roc = roc_auc_score(y_test, y_pred_proba)
pr = average_precision_score(y_test, y_pred_proba)

print("ROC-AUC:", roc)
print("PR-AUC:", pr)

ROC-AUC: 0.905183752346467
PR-AUC: 0.5878199644646995


ROC-AUC: 0.9050942599888389
PR-AUC: 0.5903242214242288


В качестве первого baseline была обучена линейная модель на текстовых признаках.
При seller-based разбиении данных достигнут ROC-AUC 0.905, PR-AUC 0.59.
Это свидетельствует о высокой информативности текстового контента карточек товаров.

In [10]:
feature_names = vectorizer.get_feature_names_out()
coef = model.coef_[0]

top_positive_idx = np.argsort(coef)[-20:]
top_negative_idx = np.argsort(coef)[:20]

print("топ слов в подделках:")
print(feature_names[top_positive_idx])

print("топ слов в оригиналах:")
print(feature_names[top_negative_idx])

топ слов в подделках:
['оперативная память' 'черный' 'клавиатура' 'аккумулятор xiaomi'
 'устройств xiaomi' 'белый' 'оригинальный' 'роутер' 'пылесос' 'pioneer'
 'картридж hp' 'геймпад' 'крышка для' 'колонка' 'монитор' 'наушники' 'кт'
 'картридж samsung' 'батарейка' 'mivis']
топ слов в оригиналах:
['для xiaomi' 'игра' 'для' 'для apple' 'gps' 'для samsung' 'galaxy'
 'realme' 'kitfort' 'samsung' 'для huawei' 'electrolux' 'infinix' 'baseus'
 'пылесоса' 'шлейф' 'xiaomi' 'для ноутбука' 'ballu' 'galaxy line']


# Baseline 2 - CatBoost
теперь оценим табличные данные кэтбустом.

In [11]:
train_df = train_df.copy()
test_df = test_df.copy()

target = "resolution"

drop_cols = [
    "id",
    "ItemID",
    "SellerID",
    "name_rus",
    "description",
    "brand_name",
    "text"
]

feature_cols = [col for col in train_df.columns if col not in drop_cols + [target]]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df[target]
y_test = test_df[target]

print("Фичей:", len(feature_cols))

Фичей: 38


In [12]:
#Заполняем только числовые пропуски нулями
num_cols = X_train.select_dtypes(include=["float64", "int64"]).columns

X_train[num_cols] = X_train[num_cols].fillna(0)
X_test[num_cols] = X_test[num_cols].fillna(0)

/var/folders/p4/qk6t1wt14b9dhc4806cz9n880000gn/T/ipykernel_85260/2675827136.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[num_cols] = X_train[num_cols].fillna(0)
/var/folders/p4/qk6t1wt14b9dhc4806cz9n880000gn/T/ipykernel_85260/2675827136.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[num_cols] = X_test[num_cols].fillna(0)


In [13]:
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", len(cat_cols))
print("Numerical:", len(num_cols))

Categorical: 1
Numerical: 37


In [14]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_cb = CatBoostClassifier(
    iterations=700,
    depth=6,
    learning_rate=0.05,
    eval_metric="AUC",
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=100
)

model_cb.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

y_pred_proba_cb = model_cb.predict_proba(X_test)[:, 1]

roc_cb = roc_auc_score(y_test, y_pred_proba_cb)
pr_cb = average_precision_score(y_test, y_pred_proba_cb)

print("CatBoost ROC-AUC:", roc_cb)
print("CatBoost PR-AUC:", pr_cb)

0:	test: 0.8765860	best: 0.8765860 (0)	total: 93ms	remaining: 1m 5s
100:	test: 0.9036785	best: 0.9045514 (72)	total: 2.68s	remaining: 15.9s
200:	test: 0.9061910	best: 0.9071260 (126)	total: 5.38s	remaining: 13.4s
300:	test: 0.9068962	best: 0.9072156 (295)	total: 8s	remaining: 10.6s
400:	test: 0.9045130	best: 0.9075768 (344)	total: 10.5s	remaining: 7.82s
500:	test: 0.9035513	best: 0.9075768 (344)	total: 13.1s	remaining: 5.2s
600:	test: 0.9030339	best: 0.9075768 (344)	total: 15.7s	remaining: 2.58s
699:	test: 0.9027946	best: 0.9075768 (344)	total: 18.5s	remaining: 0us

bestTest = 0.9075768315
bestIteration = 344

Shrink model to first 345 iterations.
CatBoost ROC-AUC: 0.9075768315271323
CatBoost PR-AUC: 0.6693931473502321


CatBoost ROC-AUC: 0.9050578499170622
CatBoost PR-AUC: 0.6230967508174646

Текст по метрикам схож с табличными данными.

# Baseline 3 - мультимодальный с эмбеддингами

In [15]:
from google.colab import files
files.upload()

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Загрузка эмбеддингов
image_emb = pd.read_csv("image_embeddings.csv")
print("Эмбеддинги загружены:", image_emb.shape)

Эмбеддинги загружены: (219195, 3)


In [ ]:
#Убедимся, что ключ называется одинаково (привести к нижнему регистру)
image_emb.columns = [c.lower() for c in image_emb.columns]

In [ ]:
print("Строк в image_emb:", len(image_emb))
print("Строк в df:", len(df))

Строк в image_emb: 219195
Строк в df (весь трейн): 197198


In [ ]:
image_emb.head()

,unnamed: 0,image_name,embedding
0,0,10.png,[-0.3904604 -0.22580202 -0.14542326 -0.032227...
1,1,100.png,[ 0.28374434 -0.14928259 0.29042315 -0.312558...
2,2,10000.png,[ 0.08850661 -0.42001283 0.4141257 -0.210395...
3,3,100009.png,[ 0.22624922 -0.3577845 0.43859392 -0.295216...
4,4,100012.png,[-0.19313276 0.09382293 -0.25836873 0.104856...


In [ ]:
#Извлекаем item_id из имени файла
image_emb['item_id'] = image_emb['image_name'].str.replace('.png', '').astype(int)

print(image_emb['item_id'].head())
print(df['ItemID'].head())  # посмотри как выглядит ItemID в df — тот же формат?

0        10
1       100
2     10000
3    100009
4    100012
Name: item_id, dtype: int64
0     78312
1    141999
2     53306
3    202599
4    163725
Name: ItemID, dtype: int64


In [ ]:
image_emb_train = image_emb[image_emb['item_id'].isin(df['ItemID'])]
print("Было в image_emb:", len(image_emb))
print("Осталось после фильтрации:", len(image_emb_train))

Было в image_emb: 219195
Осталось после фильтрации: 196460


In [ ]:
# Используем уже готовое seller-based разбиение
train_image_emb = image_emb_train[image_emb_train['item_id'].isin(train_df['ItemID'])]
test_image_emb  = image_emb_train[image_emb_train['item_id'].isin(test_df['ItemID'])]

print("Train эмбеддинги:", len(train_image_emb))
print("Test эмбеддинги:", len(test_image_emb))

Train эмбеддинги: 176646
Test эмбеддинги: 19814


In [ ]:
def parse_embedding(s):
    # убираем скобки и парсим через пробел
    s = s.strip().strip('[]')
    return np.fromstring(s, sep=' ')

X_train_img = np.vstack(train_image_emb['embedding'].apply(parse_embedding))
X_test_img  = np.vstack(test_image_emb['embedding'].apply(parse_embedding))

print("X_train_img shape:", X_train_img.shape)
print("X_test_img shape:", X_test_img.shape)

X_train_img shape: (176646, 32)
X_test_img shape: (19814, 32)


In [ ]:
# y берём через merge чтобы порядок совпадал с X
y_train_img = train_image_emb.merge(
    train_df[['ItemID', 'resolution']],
    left_on='item_id', right_on='ItemID'
)['resolution'].values

y_test_img = test_image_emb.merge(
    test_df[['ItemID', 'resolution']],
    left_on='item_id', right_on='ItemID'
)['resolution'].values

print("y_train counterfeit rate:", y_train_img.mean().round(4))
print("y_test counterfeit rate:", y_test_img.mean().round(4))
print("Размеры совпадают:", len(X_train_img) == len(y_train_img), len(X_test_img) == len(y_test_img))

y_train counterfeit rate: 0.0586
y_test counterfeit rate: 0.1319
Размеры совпадают: True True


In [ ]:
#масштабировуем
scaler = StandardScaler()
X_train_img_scaled = scaler.fit_transform(X_train_img)
X_test_img_scaled  = scaler.transform(X_test_img)

# делаем простую регрессию, но с учётом дисбаланса классов
model_img = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_img.fit(X_train_img_scaled, y_train_img)

# Метрики
y_pred_proba_img = model_img.predict_proba(X_test_img_scaled)[:, 1]

roc = roc_auc_score(y_test_img, y_pred_proba_img)
pr  = average_precision_score(y_test_img, y_pred_proba_img)

precision, recall, _ = precision_recall_curve(y_test_img, y_pred_proba_img)
mask = precision >= 0.9
recall_at_90 = recall[mask].max() if mask.any() else 0.0

print(f"Image-only ROC-AUC:        {roc:.4f}")
print(f"Image-only PR-AUC:         {pr:.4f}")
print(f"Recall @ Precision >= 0.9: {recall_at_90:.4f}")

Image-only ROC-AUC:        0.8532
Image-only PR-AUC:         0.4616
Recall @ Precision >= 0.9: 0.0042


## Выводы по baseline-моделям

Были обучены три baseline-модели на разных модальностях данных
с единым seller-based разбиением (train/test), чтобы избежать утечки данных.

**Результаты:**

| Baseline | ROC-AUC | PR-AUC | Recall@P≥0.9 |
|---|---|---|---|
| Текст (TF-IDF + LR) | 0.9051 | 0.5903 | ~0.0 |
| CatBoost (табличные признаки) | 0.9051 | 0.6231 | 0.0145 |
| Изображения (эмбеддинги + LR) | 0.8532 | 0.4616 | 0.0042 |

**Наблюдения:**

1. Текст и табличные признаки дают сопоставимый ROC-AUC (~0.90),
что говорит о высокой информативности обеих модальностей по отдельности.

2. CatBoost превосходит текстовую модель по PR-AUC (0.62 vs 0.59) —
табличные признаки лучше справляются с дисбалансом классов.

3. Изображения в текущем виде (32-мерные эмбеддинги) уступают остальным
модальностям.

4. Критическая метрика Recall@Precision≥0.9 близка к нулю у всех трёх моделей.
Это означает, что ни одна одномодальная модель не способна уверенно
детектировать контрафакт без большого числа ложных срабатываний —
что неприемлемо для production-системы модерации.

**Вывод:** одномодальные baseline-модели подтверждают информативность
каждого источника данных, но не решают задачу в полной мере.
Это мотивирует переход к мультимодальному подходу,
где фьюжн текста, табличных признаков и изображений
потенциально способен улучшить Recall@P≥0.9. Интересно будет посмотреть, какие методы принесут наилучший результат относительно этих цифр.